# DATA CLEANING & PREPROCESSING

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Load Dataset

In [2]:
df = pd.read_csv("D:\\SEM - 6\\MLDL\\ML\\ML-Cardio-Prediction\\data\\cardio_train.csv", sep=";")
print("Initial Shape:", df.shape)
df.head()

Initial Shape: (70000, 13)


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


Convert Age from Days to Years

In [3]:
df['age_years'] = (df['age'] / 365.25).astype(int)
df.drop(columns=['age'], inplace=True)

In [4]:
df.head()

,id,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years
0,0,2,168,62.0,110,80,1,1,0,0,1,0,50
1,1,1,156,85.0,140,90,3,1,0,0,1,1,55
2,2,1,165,64.0,130,70,3,1,0,0,0,1,51
3,3,2,169,82.0,150,100,1,1,0,0,1,1,48
4,4,1,156,56.0,100,60,1,1,0,0,0,0,47


Handle Invalid Blood Pressure Values

In [5]:
print("Shape befor removing outliers:", df.shape)

Shape befor removing outliers: (70000, 13)


In [6]:
df = df[(df['ap_hi'] >= 50) & (df['ap_hi'] <= 250)]
df = df[(df['ap_lo'] >= 30) & (df['ap_lo'] <= 150)]
df = df[df['ap_hi'] > df['ap_lo']]

In [7]:
print("Shape after removing outliers:", df.shape)

Shape after removing outliers: (68673, 13)


Handle Height and Weight Outliers

In [8]:
df = df[(df['height'] >= 120) & (df['height'] <= 220)]
df = df[(df['weight'] >= 30) & (df['weight'] <= 200)]

In [9]:
print("Shape after removing outliers:", df.shape)

Shape after removing outliers: (68617, 13)


# Feature Engineering : BMI Calculation

In [10]:
# Calculate BMI and add as a new feature
df['BMI'] = df['weight'] / ((df['height'] / 100) ** 2)

In [11]:
df.head()

,id,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years,BMI
0,0,2,168,62.0,110,80,1,1,0,0,1,0,50,21.967120
1,1,1,156,85.0,140,90,3,1,0,0,1,1,55,34.927679
2,2,1,165,64.0,130,70,3,1,0,0,0,1,51,23.507805
3,3,2,169,82.0,150,100,1,1,0,0,1,1,48,28.710479
4,4,1,156,56.0,100,60,1,1,0,0,0,0,47,23.011177


Remove unrealistic BMI values

In [12]:
df = df[(df['BMI'] >= 10) & (df['BMI'] <= 60)]

In [13]:
print("Shape after removing outliers:", df.shape)

Shape after removing outliers: (68594, 14)


# Encode Categorical Variables

Gender Encoding

In [14]:
df['gender'] = df['gender'].map({1: 0, 2: 1})
df.head()

,id,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years,BMI
0,0,1,168,62.0,110,80,1,1,0,0,1,0,50,21.967120
1,1,0,156,85.0,140,90,3,1,0,0,1,1,55,34.927679
2,2,0,165,64.0,130,70,3,1,0,0,0,1,51,23.507805
3,3,1,169,82.0,150,100,1,1,0,0,1,1,48,28.710479
4,4,0,156,56.0,100,60,1,1,0,0,0,0,47,23.011177


Cholesterol & Glucose Encoding (Ordinal)

In [15]:
# Maintain order while making features numeric.
df['cholesterol'] = df['cholesterol'] - 1
df['gluc'] = df['gluc'] - 1
df.head()

,id,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years,BMI
0,0,1,168,62.0,110,80,0,0,0,0,1,0,50,21.967120
1,1,0,156,85.0,140,90,2,0,0,0,1,1,55,34.927679
2,2,0,165,64.0,130,70,2,0,0,0,0,1,51,23.507805
3,3,1,169,82.0,150,100,0,0,0,0,1,1,48,28.710479
4,4,0,156,56.0,100,60,0,0,0,0,0,0,47,23.011177


Binary Lifestyle Features Validation

In [16]:
binary_cols = ['smoke', 'alco', 'active']
df[binary_cols] = df[binary_cols].astype(int)
print("Data types after conversion:", df.dtypes[binary_cols])
df[binary_cols]

Data types after conversion: smoke     int64
alco      int64
active    int64
dtype: object


,smoke,alco,active
0,0,0,1
1,0,0,1
2,0,0,0
3,0,0,1
4,0,0,0
...,...,...,...
69995,1,0,1
69996,0,0,1
69997,0,1,0
69998,0,0,0


Separate Features and Target

In [17]:
X = df.drop(columns=['cardio', 'id'])
y = df['cardio']

Train–Test Split (Stratified)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Feature Scaling

In [19]:
scaler = StandardScaler()

num_cols = [
    'age_years', 'height', 'weight',
    'ap_hi', 'ap_lo', 'BMI'
]

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

Final Shape Verification

In [20]:
print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)
X_train.head()

Train Shape: (54875, 12)
Test Shape: (13719, 12)


,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,age_years,BMI
17941,0,-0.307183,-0.638877,0.197608,-0.138799,0,0,0,0,1,-0.259858,-0.514444
26429,1,-0.560270,-0.990084,-0.400413,-0.138799,0,0,0,0,1,0.330735,-0.770252
33034,0,-1.446077,0.414745,0.197608,0.918697,0,0,0,0,1,-0.259858,1.294840
65158,0,-1.699164,-2.113947,-2.194474,-1.196296,2,2,0,0,1,1.216624,-1.566507
26934,1,0.072449,-0.919843,1.393648,0.918697,2,2,0,0,1,1.068976,-0.968662


Save Preprocessed Data

In [21]:
X_train.to_csv("../data/X_train_final.csv", index=False)
X_test.to_csv("../data/X_test_final.csv", index=False)
y_train.to_csv("../data/y_train_final.csv", index=False)
y_test.to_csv("../data/y_test_final.csv", index=False)